In [8]:
import osmnx as ox
import networkx as nx

print("Localisation et téléchargement de la Scène...")

# On donne les adresses à Python
adresse_depart = "Hanoi University of Mining and Geology, Hanoi, Vietnam"
adresse_arrivee = "Hoa Binh Park, Hanoi, Vietnam"

# On récupère les coordonnées GPS exactes
point_A = ox.geocode(adresse_depart)
point_B = ox.geocode(adresse_arrivee)

# On télécharge le réseau routier 3km autour HUMG
graphe = ox.graph_from_point(point_A, dist=3000, network_type='drive')
print(f"Réseau téléchargé ! Nœuds (intersections) trouvés : {len(graphe.nodes)}")


print("Placement de l'Acteur sur la route...")

# On cherche donc l'intersection la plus proche
noeud_origine = ox.distance.nearest_nodes(graphe, X=point_A[1], Y=point_A[0])
noeud_destination = ox.distance.nearest_nodes(graphe, X=point_B[1], Y=point_B[0])


print("Calcul du chemin le plus court...")

# shortest Path
chemin_optimal = nx.shortest_path(graphe, source=noeud_origine, target=noeud_destination, weight='length')

# calcule la distance physique (en mètres) pour notre futur calcul de CO2 !
distance_m = nx.shortest_path_length(graphe, source=noeud_origine, target=noeud_destination, weight='length')
print(f"Distance totale du trajet : {distance_m / 1000:.2f} km")

route_gdf = ox.routing.route_to_gdf(graphe, chemin_optimal)

carte_finale = route_gdf.explore(
    color="red",
    style_kwds={"weight": 6, "opacity": 0.8},
    tiles="CartoDB positron",
    tooltip="name" # afficher le nom de la rue 
)

display(carte_finale)

Localisation et téléchargement de la Scène...
Réseau téléchargé ! Nœuds (intersections) trouvés : 2474
Placement de l'Acteur sur la route...
Calcul du chemin le plus court...
Distance totale du trajet : 3.06 km


In [9]:
import osmnx as ox

print("Enrichissement de la carte (Bâtiments et Végétation)...")

# On définit ce qu'on veut télécharger depuis OpenStreetMap
tags_batiments = {'building': True}
tags_vegetation = {'leisure': 'park', 'natural': ['wood', 'scrub'], 'landuse': ['forest', 'grass']}

# On télécharge les données (ex: 1000m)
print("Téléchargement des bâtiments en cours...")
batiments_gdf = ox.features_from_point(point_A, tags=tags_batiments, dist=1000)

print("Téléchargement de la végétation en cours...")
vegetation_gdf = ox.features_from_point(point_A, tags=tags_vegetation, dist=1000)

print("Données environnementales récupérées !")

# Création de la carte
print("Génération de la carte complète...")

# On met la végétation en vert
carte_complete = vegetation_gdf.explore(
    color="lightgreen",
    style_kwds={"fillOpacity": 0.5, "weight": 0},
    tiles="CartoDB positron",
    name="Végétation (Absorption CO2)"
)

# On ajoute les bâtiments en gris
batiments_gdf.explore(
    m=carte_complete,
    color="gray",
    style_kwds={"fillOpacity": 0.7, "weight": 1},
    name="Bâtiments (Obstacles au Vent)"
)

# On rajoute ton trajet en rouge
route_gdf.explore(
    m=carte_complete,
    color="red",
    style_kwds={"weight": 6, "opacity": 0.8},
    tooltip="name",
    name="Trajet du véhicule"
)

# On ajoute un menu pour pouvoir cocher/décocher les couches
import folium
folium.LayerControl().add_to(carte_complete)

display(carte_complete)

Enrichissement de la carte (Bâtiments et Végétation)...
Téléchargement des bâtiments en cours...
Téléchargement de la végétation en cours...
Données environnementales récupérées !
Génération de la carte complète...


In [10]:
import numpy as np
import geopandas as gpd
import folium

print(" Modélisation mathématique du Vent...")

# Définition de la Météo
angle_vent_degres = 270 # Vent soufflant vers le Nord-Est
vitesse_vent = 20      # Force du vent

angle_rad = np.radians(angle_vent_degres)

# Création du Panache de pollution
zone_initiale = route_gdf.geometry.buffer(0.00015)
decalage_x = np.cos(angle_rad) * (vitesse_vent * 0.00001)
decalage_y = np.sin(angle_rad) * (vitesse_vent * 0.00001)

panache_geometry = zone_initiale.translate(xoff=decalage_x, yoff=decalage_y)
panache_gdf = gpd.GeoDataFrame(geometry=panache_geometry, crs=route_gdf.crs)

print("Filtrage des bâtiments et Calcul de l'Effet Canyon Urbain...")

# On filtre pour éviter l'erreur
batiments_polygones = batiments_gdf[batiments_gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]

# L'Effet Canyon
pollution_stagnante = gpd.overlay(panache_gdf, batiments_polygones, how='intersection')

print("Génération de la carte de dispersion...")

# Les bâtiments en gris
carte_dispersion = batiments_polygones.explore(
    color="gray", style_kwds={"fillOpacity": 0.3, "weight": 1}, name="Bâtiments"
)

# La route (Source CO2) en noir
route_gdf.explore(
    m=carte_dispersion, color="black", style_kwds={"weight": 4}, name="Route (Source CO2)"
)

# Le nuage poussé par le vent en orange transparent
panache_gdf.explore(
    m=carte_dispersion, color="orange", style_kwds={"fillOpacity": 0.4, "weight": 0}, name="Dispersion par le vent"
)

# Le danger (Effet Canyon) en rouge vif !
if not pollution_stagnante.empty:
    pollution_stagnante.explore(
        m=carte_dispersion, color="red", style_kwds={"fillOpacity": 0.9, "weight": 2}, name="CO2 Stagnant (Danger)"
    )

folium.LayerControl().add_to(carte_dispersion)

display(carte_dispersion)

 Modélisation mathématique du Vent...
Filtrage des bâtiments et Calcul de l'Effet Canyon Urbain...
Génération de la carte de dispersion...


/tmp/ipykernel_176050/266358479.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zone_initiale = route_gdf.geometry.buffer(0.00015)
